In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import numpy as np
from pathlib import Path

In [ ]:
print('x')

### read data matrix from path

In [ ]:
results_dir = Path(os.environ["EPHEMERAL"]) / "pyspoc_runs/results/concat/"

N = 1000
P = 100

dataset_folder = f'syn_data_N{N}_P{P}_num_seed_10'

dm_file = 'datamatrix__3configs__filtered_st85_dt25.parquet'

data_path = results_dir / dataset_folder / dm_file

In [ ]:
data_matrix = pd.read_parquet(data_path)

## OPTIONAL: REMOVE PATHS FROM DATA MATRIX COLUMN NAMES
## (EXTERNAL MODULES GET SAVED WITH THE FULL PATH)

stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

data_matrix_names = data_matrix.columns.str.replace(stats_path, ".", regex=False)
data_matrix.columns = data_matrix_names

print(data_matrix.shape[0],data_matrix.shape[1])

In [ ]:
# pd.Series of short col names so they're easier to read
short_stat_names = pd.Series(data_matrix.columns.str.split(".").str[-4:-1].str.join("."), index=data_matrix.columns)

In [ ]:
print('the following cols. are constant across all datasets:')
num_unique_val_per_stat = data_matrix.nunique()
print(short_stat_names[num_unique_val_per_stat <= 1])

# remove constant columns
print('removing constant cols')
df_processed = data_matrix.loc[:,~(num_unique_val_per_stat <= 1)].copy()

#### examining duplicate columns

In [ ]:
T = df_processed.T
dups = T[T.duplicated(keep=False)] # all columns that have a copy
groups = [[len(list(g.index))] + list(g.index) for _, g in dups.groupby(list(dups.columns), dropna=False)]
groups_short = [[len(list(g.index))] + list(short_stat_names[g.index]) for _, g in dups.groupby(list(dups.columns), dropna=False)]

## ---- FASTER WAY TO FIND DUPS, HASH THE ENTIRE COL
# h = df_processed.apply(lambda c: hash(c.to_numpy().tobytes())) # one hash per column
# groups = [list(cols) for cols in h.groupby(h).groups.values() if len(cols) > 1]
# for g in groups[:10]:
#     print(g)

print(len(groups), "groups of identical columns")
df_dup_cols = pd.DataFrame(groups_short)

In [ ]:
df_dup_cols.sort_values(by=0,ascending=True)
# .to_csv(f'duplicate_cols_{dataset_folder}.csv')

the duplicate cols both measure the same thing: L2 norm of a matrix is its largest singular value


In [ ]:
df_dedup = df_processed.T.drop_duplicates().T.copy()

In [ ]:
# CURRENTLY I MANUALLY MAKE THE DETERMINNANT LOG
# TO BE IMPLEMENTED IN LIBRARY
detCols = [v for v in df_dedup.columns if 'pyspoc.reducers.basic.Determinant.scaled' in v]

logdet_per_dim = (np.log(df_dedup[detCols].abs().clip(lower=1e-300))) / P
sign = np.sign(df_dedup[detCols])
df_dedup[detCols] = logdet_per_dim

In [ ]:
df_processed.shape[1]-len(groups) == df_dedup.shape[1]

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
(df_dedup.max(axis=0)-df_dedup.min(axis=0)).describe()

In [ ]:
# df_dedup.abs().max().sort_values(ascending=False)

In [ ]:
# CHECK FOR EXTREMELY BIG VALUES:
# mx = df_dedup.max().sort_values(ascending=False)
# print(mx.head(15))
# big = df_dedup.columns[df_dedup.max() > 1e40]
# print(len(big), "columns above 1e40")
# print(df_dedup[big].idxmax().value_counts().head())

In [ ]:
print("NaN:", np.isnan(df_dedup).sum().sum(), " inf:", np.isinf(df_dedup).sum().sum(), " max |x|:", np.nanmax(np.abs(df_dedup)))

In [ ]:
dataset_config_csv= pd.read_csv(Path('hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/metadata_files').resolve() / f'{dataset_folder}.csv')

In [ ]:
safe_id_setting_map = dataset_config_csv.set_index('safe_data_id')['setting'].to_dict()

In [ ]:
# df_dedup.groupby(did_seed_concat).mean().var(axis=0)
# df_dedup.groupby(did_seed_concat).var().mean(axis=0)

In [ ]:
# long

In [ ]:
# rough_icc[rough_icc>0.999]

In [ ]:
# https://github.com/raphaelvallat/pingouin/blob/main/src/pingouin/reliability.py#L173-L427
did_seed_concat = df_dedup.index.map(safe_id_setting_map)
between = df_dedup.groupby(did_seed_concat).mean().var(axis=0)
within = df_dedup.groupby(did_seed_concat).var().mean(axis=0)
rough_icc = (between / (between + within)).sort_values()
rough_icc.plot(kind='hist')

# ICC(1) USING ANOVA-1 WAY 
# from sklearn.feature_selection import f_classif
# g = pd.Series(did_seed_concat, index=df_dedup.index)
# multi = g.map(g.value_counts()) >= 2
# X, groups = data_matrix[multi], g[multi]
# F, p = f_classif(X, groups)
# n_i = groups.value_counts()
# N, n_groups = len(groups), len(n_i)
# k0 = (N - (n_i ** 2).sum() / N) / (n_groups - 1)
# icc1 = pd.Series((F - 1) / (F + k0 - 1), index=data_matrix.columns).sort_values()
# # icc1.plot(kind='hist')

In [ ]:
# near1 = rough_icc.index[rough_icc > 0.99]
# print(len(near1), "of", len(rough_icc), "statistics")
# within_sd = df_dedup[near1].groupby(did_seed_concat).std().max()
# print((within_sd == 0).sum(), "are exactly constant within every setting")
# for c in near1[:10]:
#     print(c.split("statistics.")[-1][:60])
        #   data_matrix[c].corr(meta["n_samples"], method="spearman").round(2),
        #   data_matrix[c].corr(meta["n_dim"], method="spearman").round(2))

## downstream analyses on the data matrix

#### preprocessing / running trial stuff

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
# nb_dir = Path(globals()["__vsc_ipynb_file__"]).parent

nb_dir = Path("hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/py_scripts/dataviz_notebooks/")
FIG_DIR = nb_dir.resolve() / 'figures'
os.makedirs(FIG_DIR,exist_ok=True)

In [ ]:
def fine_group(v):
    return v.split('__')[0]

def coarse_group(v):
    if v.startswith("GMM_"):
        return "cluster_gmm"
        # if v.startswith("GMM_iso"):
        #     return "cluster_gmm_iso"
        # if v.startswith("GMM_aniso"):
        #     return "cluster_gmm_aniso"    
    if v.startswith("cpm_counts_"):
        return "count_cpm"
    if v.startswith("counts_"):
        return "count_raw"
    if v.startswith("gp_lmm_"):
        return "manifold_lmm"
    if "hypercube_grid" in v:
        return "cluster_grid"
    if v.endswith("gaussian"):
        # if v.startswith("iso"):
        #     return "iso_gaussian"
        # if v.startswith("aniso"):
        #     return "aniso_gaussian"        
        return "gaussian"
    if v.startswith("concentric_shell"):
        return "concentric_shell"
    return v

# print(coarse.value_counts())

In [ ]:
df_module_cluster = df_dedup.sample(frac=1,random_state=84372893)

In [ ]:
# df_module_cluster.index.map(fine_group).value_counts()

In [ ]:
df_module_cluster.index.map(fine_group).map(coarse_group).value_counts()

In [ ]:
# df_module_cluster.index

In [ ]:
X = StandardScaler().fit_transform(df_module_cluster)

n_pcs = 20
pca = PCA(n_components=n_pcs)
Z = pca.fit_transform(X)

In [ ]:
coarse = df_module_cluster.index.map(fine_group).map(coarse_group)

In [ ]:
explained_var = pca.explained_variance_
explained_var_ratio = pca.explained_variance_ratio_

#### pca and tsne viz

In [ ]:
n_pcs_plot=10
xv = [v+1 for v in range(n_pcs_plot)]
plt.plot(xv,pca.explained_variance_ratio_[:n_pcs_plot],marker='o')
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
plt.xticks(xv)

figname = f'PCA_explainedVar_{n_pcs_plot}PCs_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fine = df_module_cluster.index.map(fine_group)
coarse = fine.map(coarse_group)

group_cmaps = {
    "cluster_gmm": "Blues",
    "cluster_grid":"Purples",
    "manifold_lmm":"Greens",
    "count_cpm":"Oranges",
    "gaussian":"Reds",
    "concentric_shell":"Greys",
}

fams = pd.Series(coarse.to_numpy(), index=fine.to_numpy()).groupby(level=0).first()
palette = {f: c for g, cm in group_cmaps.items()
           for f, c in zip(sorted(fams.index[fams == g]),
                           sns.color_palette(cm, (fams == g).sum() + 2)[2:])}
hue_order = sorted(palette, key=lambda f: (fams[f], f))

fams_c = pd.Series(coarse.to_numpy(), index=coarse.to_numpy()).groupby(level=0).first()
palette_c = {f: c for g, cm in group_cmaps.items()
           for f, c in zip(sorted(fams_c.index[fams_c == g]),
                           sns.color_palette(cm, (fams_c == g).sum() + 2)[2:])}
hue_order_c = sorted(palette_c, key=lambda f: (fams_c[f], f))


In [ ]:
n_pcs_plot=4
pc_names = [f"PC{i + 1}" for i in range(n_pcs_plot)]
Z_pl = Z[:,:n_pcs_plot]

pc_df = pd.DataFrame(Z_pl, columns=pc_names)
pc_df["fine"] = fine.to_numpy()
pc_df["coarse"] = coarse.to_numpy()

# --- pairplot ---

g = sns.PairGrid(pc_df, x_vars=pc_names[:-1], y_vars=pc_names[1:], hue="coarse",
                 hue_order=hue_order_c, palette=palette_c, height=2.2, corner=True)
g.map_offdiag(sns.scatterplot, s=8, alpha=0.7, linewidth=0)
g.add_legend()
sns.move_legend(g, "upper right", bbox_to_anchor=(0.98, 0.98), ncol=2,
                title=None, frameon=False, fontsize=8, markerscale=1.5)
g.legend.set_title(None)
plt.tight_layout()
figname = f'PCA_coarse_{n_pcs_plot}PCs_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

In [ ]:
loadings.shape

In [ ]:
# xticks = [f'PC{i+1}' for i in range(len(pca.components_))]
# yticks = ['' for i in range(df_module_cluster.shape[1])]
# plt.figure(figsize=(12, 8))
# sns.heatmap(loadings, annot=False, cmap='coolwarm', xticklabels=xticks, yticklabels=yticks)
# sns.heatmap(pca.components_.T, annot=False, cmap='coolwarm', xticklabels=xticks, yticklabels=yticks)

In [ ]:
# pca.components_

In [ ]:
from sklearn.manifold import TSNE

X = StandardScaler().fit_transform(df_module_cluster)

E = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(X)

In [ ]:
coarse.unique()

In [ ]:
plot_df = pd.DataFrame({"tSNE1": E[:, 0], "tSNE2": E[:, 1], "fine": fine.to_numpy(), "coarse":coarse.to_numpy()})

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=plot_df, x="tSNE1", y="tSNE2", hue="fine",
                hue_order=sorted(palette, key=lambda f: (fams[f], f)),
                palette=palette, s=12, alpha=0.8, linewidth=0)
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=plot_df, x="tSNE1", y="tSNE2", hue="coarse", s=12, alpha=0.8, linewidth=0,palette=palette_c)
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")

#### regression / classification to assess separability by features

In [ ]:
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline

In [ ]:
# names

In [ ]:
X = df_module_cluster

groups = df_module_cluster.index.map(fine_group)
coarse = df_module_cluster.index.map(fine_group).map(coarse_group)

y, names = pd.factorize(coarse)
y_fine, names_fine = pd.factorize(groups)

pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

In [ ]:
# leave one y_fine group out for each cv split
cv_loo_pred = cross_val_predict(pipe, X = X, y = y, groups=y_fine, cv=LeaveOneGroupOut().split(X, y, groups=y_fine))

In [ ]:
print('accuracy over all coarse labels: ')
print((cv_loo_pred==y).mean(), '+-', (cv_loo_pred==y).std())

In [ ]:
y_fine_name_dict = {i : names_fine[i] for i in range(len(names_fine))}
y_c_name_dict = {i : names[i] for i in range(len(names))}

In [ ]:
res = pd.DataFrame({"y_fine": np.asarray(y_fine), "y_coarse": np.asarray(y), "pred": np.asarray(cv_loo_pred)})
n_fine = res["y_fine"].value_counts()

wrong = res[res["y_coarse"] != res["pred"]]
top_wrong = wrong.groupby("y_fine")["pred"].agg(lambda s: s.value_counts().idxmax())

In [ ]:
# wrong

In [ ]:
all_wrong = wrong.groupby("y_fine")["pred"].agg(lambda s: ", ".join(f"{y_c_name_dict[k]}" for k, v in s.value_counts().items()))
all_wrong.index = all_wrong.index.map(y_fine_name_dict)

In [ ]:
n_fine = res["y_fine"].value_counts()
n_fine.index = n_fine.index.map(y_fine_name_dict)

In [ ]:
mean_per_fine = pd.Series(cv_loo_pred == y, index=y_fine).groupby(level=0).agg(['mean','std'])
mean_per_fine.index = names_fine
mean_per_fine=mean_per_fine.sort_index()
mean_per_fine['misclassified_labels'] = all_wrong
mean_per_fine['num_total'] = n_fine
mean_per_fine.sort_values(by=['mean','misclassified_labels'])

In [ ]:
ct = pd.crosstab(pd.Series(np.asarray(y_fine), name="held-out family"),
                 pd.Series(np.asarray(cv_loo_pred), name="predicted group"), normalize="index")

ct.index = ct.index.map(y_fine_name_dict) + ', size:' + mean_per_fine['num_total'].astype(str)
ct.columns = ct.columns.map(y_c_name_dict)

In [ ]:
fig, ax = plt.subplots(figsize=(1.1 * ct.shape[1] + 4, 0.35 * ct.shape[0] + 2))
sns.heatmap(ct, annot=True, cmap="Blues", vmin=0, vmax=1,
            cbar_kws={"label": "fraction of family"}, linewidths=0.5, ax=ax,
            annot_kws={"fontsize": 10})
ax.set_xlabel("predicted y_coarse"); ax.set_ylabel("held out y_fine family")

from matplotlib.patches import Rectangle
for i, fam in enumerate(ct.index):
    coarse_val = ct.index.map(coarse_group)[i]
    if coarse_val in ct.columns:
        ax.add_patch(Rectangle((list(ct.columns).index(coarse_val), i), 1, 1, fill=False, ec="red", lw=1.5))
# ax.set_yticks()
plt.show()

### examining statistic modules

Dist_matrix → spearman corr → dissimilarity mtx (D) → kNN 

Each column of D is a full pyspoc statistic+reducer or rstatistic

→ Shorten the full names to get base classes 

→ examine statistics where neighbourhood sets have different base classes

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
corr = df_module_cluster.corr(method='spearman')
abs_corr = corr.abs()
dissim = np.array(1-abs_corr)

In [ ]:
k = 5
nn = NearestNeighbors(n_neighbors=k + 1, metric="precomputed").fit(dissim)

In [ ]:
# short_stat_names = pd.Series(df_module_cluster.columns.str.split(".").str[-4:-1].str.join("."), index=df_module_cluster.columns)

In [ ]:

# col_short_map = {v : v.split('.')[-6] + '.' + v.split('.')[-2] if len(v.split('.'))>6 else v.split('.')[-2]
#                                                   for v in list(abs_corr.columns)}

col_short_map = {v : v.split('.')[-2] if len(v.split('.'))>6 else v.split('.')[-2]
                                                  for v in list(abs_corr.columns)}

In [ ]:
dist, idx = nn.kneighbors(dissim)

names = abs_corr.columns
short_names = abs_corr.columns.map(col_short_map)

nb_df = pd.DataFrame({
    "stat": names,
    "stat_short" : short_names,
    "neighbor_full": [list(names[i[1:]]) for i in idx],    
    "neighbor_set": [set(short_names[i[1:]]) for i in idx],
    "neighbor_len": [len(set(short_names[i[1:]])) for i in idx],    
    "mean_abs_corr": 1 - dist[:, 1:].mean(axis=1),
}).sort_values("mean_abs_corr")

In [ ]:
nb_df['neighbor_len'].quantile(0.9)

In [ ]:
short_names.unique()

In [ ]:
# stat_short_type = 'DimReductionStatistics'
nb_len_low = 2

for stat_short_type in ['DimReductionStatistics','CentroidDistanceDistributionCompare']:

    sub_nb_df = nb_df[(nb_df['stat_short']==stat_short_type) & (nb_df['neighbor_len']>nb_len_low)].sort_values(by='neighbor_len')
    
    if sub_nb_df.shape[0]>0:
        
        print('STATISTIC CLASS:')
        print(stat_short_type)
        print(sub_nb_df.shape[0])
        
        for ix,row in sub_nb_df.iterrows():
            
            # if 0.9 > row['mean_abs_corr'] > 0.8:
            
            print('\tSTATISTIC:')
            print('\t'+row['stat'])
            print('\t'+str(row['mean_abs_corr']))
            print('\t\tNBS:')
            for nb_nm in row['neighbor_full']:
                print('\t\t\t'+nb_nm)


In [ ]:
# nb_df.sort_values(by='mean_abs_corr',ascending=False)

In [ ]:
# MOVED TO GOOGLE COLAB FOR INTERACTIVE VIZ
# import networkx as nx

# dist, idx = nn.kneighbors(dissim)
# names = abs_corr.columns

# G = nx.Graph()
# for i, row in enumerate(idx):
#     for j, d in zip(row[1:], dist[i, 1:]):
#         sim_wt = 1 - d
#         # if 0.8 > sim_wt > 0.7:
#         G.add_edge(names[i], names[j], weight=sim_wt)

# pos = nx.spring_layout(G, weight="weight", seed=0, k=0.15)
# fig, ax = plt.subplots(figsize=(5, 5))
# nx.draw_networkx_edges(G, pos, alpha=0.15, width=0.5, ax=ax)
# import pickle

# with open(FIG_DIR / f"{dataset_folder}_stat_knn_graph.pkl", "wb") as f:
#     pickle.dump({"G": G, "dist": dist,"idx":idx,"dissim":dissim}, f)

## other code / archived for now

### logistic regression archived

In [ ]:
# mean_per_coarse = pd.Series(cv_loo_pred == y, index=y).groupby(level=0).agg(['mean','std'])
# mean_per_coarse.index = names
# mean_per_coarse

In [ ]:
# could do StratifiedGroupKFold otherwise:
# list(StratifiedGroupKFold(n_splits=10).split(X,y,groups=y_fine))
# LeaveOneGroupOut().split(X, y, groups=y_fine)

In [ ]:
# check separability of each statistic
# from sklearn.tree import DecisionTreeClassifier
# single = {c: cross_val_score(DecisionTreeClassifier(max_depth=3), X[[c]], y, cv=20).mean()
#         for c in X.columns}
# print(pd.Series(single).sort_values(ascending=False).head(10))

In [ ]:
# pred = cross_val_predict(LogisticRegression(max_iter=5000), X=X, y=y, cv=LeaveOneGroupOut(),groups=groups)
# (pred==y).mean()
# coarse.unique()

In [ ]:
# print(pd.crosstab(family, pred, rownames=["held-out family"], colnames=["predicted"]))

In [ ]:
# print(f"accuracy {acc.mean():.2f} +/- {acc.std():.2f}, chance {groups.value_counts(normalize=True).max():.2f}")

In [ ]:
# setting = data_matrix.index.str.rsplit("__seed", n=1).str[0]
# acc = cross_val_score(LogisticRegression(max_iter=5000), X, y, groups=setting, cv=GroupKFold(n_splits=5))
# setting.value_counts().value_counts()
# print(f"accuracy {acc.mean():.2f} +/- {acc.std():.2f}, chance {groups.value_counts(normalize=True).max():.2f}")

### other archived

In [ ]:
stat_short_name = pd.Series(data_matrix.columns.str.split(".").str[-4:].str.join("."), index=data_matrix.columns)

In [ ]:
# corr_mtx = data_matrix.corr(method="spearman")
# D = (1 - corr_mtx.abs()).to_numpy()
# np.fill_diagonal(D, 0)

# corr_link= linkage(squareform(1-abs_corr,checks=False),method='average')

In [ ]:
# find clusters/modules in the data
modules = pd.Series(fcluster(corr_link,t=0.05,criterion='distance'), index=abs_corr.index)

In [ ]:
print("found " , modules.nunique(), "modules out of ", df_module_cluster.shape[1], " features")
print(f'module id with size..\n{modules.value_counts()[:3]}')
# modules.value_counts().value_counts().sort_values(ascending=False)

In [ ]:
for m in modules.value_counts().index[:5]:
    members = modules.index[modules == m]
    print(f"module {m} ({len(members)} features):")
    print(stat_short_name[members].value_counts(), "\n")

In [ ]:
plt.figure(figsize=(12, 8))
sns.clustermap(df_module_cluster.fillna(0), cmap="RdBu_r", center=0, vmin=-3, vmax=3,
               xticklabels=False, yticklabels=False, figsize=(12, 10))

plt.xlabel("statistics"); plt.ylabel("datasets")
plt.show()

In [ ]:
# n_pcs_plot = 5
# pairs = [(i,i+1) for i in range(n_pcs_plot)]

# ncols = int(np.ceil(len(pairs) / 2))
# fig, axes = plt.subplots(2, ncols, figsize=(5 * ncols, 10))
# axes = np.ravel(axes)

# for ax, (a, b) in zip(axes, pairs):
#     for g in sorted(coarse.unique()):
#         m = (coarse == g)
#         ax.scatter(Z[m, a], Z[m, b], s=10, alpha=0.7, label=g)
#     ax.set_xlabel(f"PC{a + 1} ({explained_var_ratio[a]:.0%})")
#     ax.set_ylabel(f"PC{b + 1} ({explained_var_ratio[b]:.0%})")

# for ax in axes[len(pairs):]:
#     ax.set_visible(False)

# handles, labels = axes[0].get_legend_handles_labels()
# fig.legend(handles, labels, fontsize=8, markerscale=1.5, loc="center left", bbox_to_anchor=(1.0, 0.5))

# fig.tight_layout()
# figname = f'PCA_coarse_{n_pcs_plot}PCs_{dataset_folder}'
# fig.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
sns.heatmap(df_module_cluster.fillna(0), cmap="RdBu_r", center=0, vmin=-3, vmax=3,
               xticklabels=False, yticklabels=False)

plt.xlabel("statistics"); plt.ylabel("datasets")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

corr = df_module_cluster.corr(method='spearman')
abs_corr = corr.abs()

dissim_link= linkage(squareform(1-abs_corr,checks=False),method='average')

In [ ]:
# find clusters/modules in the data
modules = pd.Series(fcluster(dissim_link,t=0.2,criterion='distance'), index=abs_corr.index)

In [ ]:
modules.value_counts().value_counts().sort_values(ascending=False)

In [ ]:
print("found " , modules.nunique(), "modules out of ", df_module_cluster.shape[1], " features")
print(f'module id with size..\n{modules.value_counts()[:3]}')
print(f'module size counts\n{modules.value_counts().value_counts()}')

In [ ]:
pyspoc_stat_fam = pd.Series(data_matrix.columns.str.split(".").str[-4:].str.join("."), index=data_matrix.columns)

In [ ]:
for m in modules.value_counts().index[:5]:
    members = modules.index[modules == m]
    print(f"module {m} ({len(members)} features):")
    print(pyspoc_stat_fam[members].value_counts(), "\n")

In [ ]:
# code to look at spearman correlations
# corr = df_dedup.corr(method="spearman")
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
# pairs = upper.stack().sort_values(key=abs, ascending=False)
# exact = pairs[pairs.abs() == 1.0]
# print(len(exact), "pairs with corr = 1")
# print(exact.head(30))

In [ ]:
# OLD: CHANGED NOW
# # the data matrix is quite bloated with 'Moment' features right now
# # i think that reducer needs to be changed. im just dropping it for now.

# is_moment_vec = data_matrix.columns.str.contains(r"Moment\.\w+_\d+$", regex=True)
# df_no_moment = data_matrix.loc[:, ~is_moment_vec]
# print(f"dropped {is_moment_vec.sum()} per-column moment features")

In [ ]:
# sp.moment(cols,axis=1,order=[1,2,3]).mean(axis=1)
# n = data.shape[0]
# cols = data[~np.eye(n, dtype=bool)].reshape(n, n - 1)     # row i = column i of data, without its diagonal entry
# import numpy as np
# import scipy.stats as sp

# rng = np.random.default_rng(0)
# data = rng.standard_normal((100, 100))   # stand-in for a 5 x 4 matrix

# print(sp.moment(data, order=[1]))  # default axis=0: one value per column
# print(sp.moment(data, order=[1], axis=None))  # whole matrix: one value
# print(sp.moment(data, order=[1, 2, 3]).mean(axis=1))

In [ ]:
# https://dataknowsall.com/blog/pcavisualized.html

# def transform_pca(X, n):

#     pca = PCA(n_components=n)
#     pca.fit(X)
#     X_new = pca.inverse_transform(pca.transform(X))

#     return X_new

# rows = 5
# cols = 5
# comps = 1

# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# fig, axes = plt.subplots(rows, 
#                          cols, 
#                          figsize=(12,12), 
#                          sharex=True, 
#                          sharey=True)


# for row in range(rows):
#     for col in range(cols):
#         try:
#             X_new = transform_pca(X_scaled, comps)
#             ax = sns.scatterplot(x=X_scaled[:, 0], 
#                                  y=X_scaled[:, 1], 
#                                  ax=axes[row, col], 
#                                  color='grey', 
#                                  alpha=.3)
#             ax = sns.scatterplot(x=X_new[:, 0], 
#                                  y=X_new[:, 1], 
#                                  ax=axes[row, col], 
#                                  color='black')
            
#             # for g in sorted(coarse.unique()):
#             #     m = (coarse == g)
#             #     ax.scatter(X_new[m, 0], X_new[m, 1], ax=axes[row, col], s=10, alpha=0.7, label=g)
            
#             ax.set_title(f'PCA Components: {comps}');

#             comps += 1
#         except:
#             pass
# plt.tight_layout()
# # plt.savefig('pcavisualize_2.png', dpi=300)